### SARIMA Forecast
### Honours Project
### Date last updated: 15/01/2026
### Author: Liam Rooney
### Model: v2.1

Let's first import the packages required to create an XGBoost model to forecast time-series predictions.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import glob
import time
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from plotly.offline import init_notebook_mode, iplot
import plotly.graph_objs as go
from plotly import tools
init_notebook_mode(connected=True)
from sklearn.metrics import mean_squared_error

# Set working directory
import os

os.chdir("/PHI_conf/PrescribingBCS/Topics/Budgets/Phasings/Development/prescribing-forecast")
print(f"Changed working directory to: {os.getcwd()}")

Changed working directory to: /PHI_conf/PrescribingBCS/Topics/Budgets/Phasings/Development/prescribing-forecast


We can then read the data in and wrangle it to a format that is compatible with the models we are looking to create.

In [2]:
# Read in data and filter for NHS A&A
df = pd.read_csv("shiny/forecasts/data/Historical Data.csv", thousands=',')

df = df[df['Disp Health Board Name'] == 'NHS AYRSHIRE & ARRAN']

# Check filter has worked
df.head()

# Drop board column
del df['Disp Health Board Name']

df.head()

,Paid Date,Claim PD Number of Paid Items,Claim PD Paid GIC excl. BB
1,30/04/2009,544244,5937827.47
2,31/05/2009,534859,5844378.81
3,30/06/2009,562027,6126432.39
4,31/07/2009,582092,6401860.72
5,31/08/2009,533418,5891049.14


In [3]:
# Check for missing values
df.isnull().sum()

Paid Date                        0
Claim PD Number of Paid Items    0
Claim PD Paid GIC excl. BB       0
dtype: int64

In [4]:
# Check datatype of each column
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, 1 to 201
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Paid Date                      201 non-null    object 
 1   Claim PD Number of Paid Items  201 non-null    int64  
 2   Claim PD Paid GIC excl. BB     201 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 6.3+ KB


With the relevant pre-processing procedures complete, we can begin to properly prepare the data for modelling. First, let's convert the 'Paid Date' column into the pandas datetime format and create features (also known as "feature engineering").

In [5]:
df['time'] = pd.to_datetime(df['Paid Date'])
df['year'] = df.time.dt.year
df['month'] = df.time.dt.month
df.drop('Paid Date', axis = 1, inplace = True)

# Move time column to front
col = df.pop('time')
df.insert(0, 'time', col)

df.head()

/tmp/ipykernel_623/2086475796.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['time'] = pd.to_datetime(df['Paid Date'])


,time,Claim PD Number of Paid Items,Claim PD Paid GIC excl. BB,year,month
1,2009-04-30,544244,5937827.47,2009,4
2,2009-05-31,534859,5844378.81,2009,5
3,2009-06-30,562027,6126432.39,2009,6
4,2009-07-31,582092,6401860.72,2009,7
5,2009-08-31,533418,5891049.14,2009,8


In [6]:
# Sort the df using the paid date column and then drop it
df = df.sort_values(by = 'time')

df.head()

,time,Claim PD Number of Paid Items,Claim PD Paid GIC excl. BB,year,month
1,2009-04-30,544244,5937827.47,2009,4
2,2009-05-31,534859,5844378.81,2009,5
3,2009-06-30,562027,6126432.39,2009,6
4,2009-07-31,582092,6401860.72,2009,7
5,2009-08-31,533418,5891049.14,2009,8


In [7]:
del df['time']
df.head()

,Claim PD Number of Paid Items,Claim PD Paid GIC excl. BB,year,month
1,544244,5937827.47,2009,4
2,534859,5844378.81,2009,5
3,562027,6126432.39,2009,6
4,582092,6401860.72,2009,7
5,533418,5891049.14,2009,8


To help capture trend and seasonality within the model, lag features of the target variable can be added as features.

In [8]:
# We will add lags for the previous month, previous end of quarter (3 months ago), and previous year (same month)
df['lag1'] = df['Claim PD Number of Paid Items'].shift(1)
df['lag3'] = df['Claim PD Number of Paid Items'].shift(3)
df['lag12'] = df['Claim PD Number of Paid Items'].shift(12)

df[:14]

,Claim PD Number of Paid Items,Claim PD Paid GIC excl. BB,year,month,lag1,lag3,lag12
1,544244,5937827.47,2009,4,NaN,NaN,NaN
2,534859,5844378.81,2009,5,544244.0,NaN,NaN
3,562027,6126432.39,2009,6,534859.0,NaN,NaN
4,582092,6401860.72,2009,7,562027.0,544244.0,NaN
5,533418,5891049.14,2009,8,582092.0,534859.0,NaN
6,549080,6055224.87,2009,9,533418.0,562027.0,NaN
7,576149,6259407.51,2009,10,549080.0,582092.0,NaN
8,529570,5699805.60,2009,11,576149.0,533418.0,NaN
9,610563,6675456.41,2009,12,529570.0,549080.0,NaN
10,520856,5606464.68,2010,1,610563.0,576149.0,NaN


In [9]:
# If there is categorical data present, convert it using pd.get_dummies(df)
# df1 = pd.get_dummies(df)

To build supervised ML models, we need to consider the features that are significant to the target feature. 

In [10]:
# Drop NAs to create feature scores
df1 = df.dropna()

x = df1.drop(columns = ['Claim PD Number of Paid Items'])
y = df1.iloc[:, 0]

# Use SelectKBest to create feature significance scores
st=time.time()
bestfeatures = SelectKBest(score_func=f_regression)
fit = bestfeatures.fit(x,y)
et=time.time()-st
print(et)
dfscores = pd.DataFrame(fit.scores_)
dfcolumns = pd.DataFrame(x.columns)
featureScores = pd.concat([dfcolumns,dfscores],axis=1)
featureScores.columns = ['Featuress','Score']
best_features=featureScores.nlargest(5,'Score')
best_features

0.04261946678161621


/PHI_conf/PrescribingBCS/Topics/Budgets/Phasings/Development/prescribing-forecast/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10 is greater than n_features=6. All the features will be returned.
  warnings.warn(


,Featuress,Score
5,lag12,1093.268341
0,Claim PD Paid GIC excl. BB,1015.404532
1,year,590.675411
4,lag3,475.466872
3,lag1,236.087718


Each model should split its dataframe into three parts when looking to model data - training, testing and validation sets. The training set should be approx. 70%, testing 20% and validation 10%.

In [11]:
# Find number of rows, create variables for 70%, 20% and 10%
total_rows = len(df1)
print("Dataframe contains", total_rows, "rows.\n")

# Create number of rows needed for training
training_rows = round(total_rows * 0.7)

# Number of rows for testing
test_rows = total_rows - training_rows

test1_rows = round(test_rows * 2/3)

vali_rows = test_rows - test1_rows

# Now let's test if these match up
if total_rows == training_rows + test1_rows + vali_rows:
    print("Calculated rows for train-test-validation split MATCHES total dataframe rows")
else:
    print("Calculated rows for train-test-validation split DOES NOT match total dataframe rows")


Dataframe contains 189 rows.

Calculated rows for train-test-validation split MATCHES total dataframe rows


With the row counts verified, we can now create the data splits.

In [12]:
# train-test-validation splits
test = df1.tail(test1_rows + vali_rows)

# test dataset
test1 = test.head(test1_rows)

# training set
train = df1.head(training_rows)

# validation set
pred = test.tail(vali_rows)

In [13]:
# Create target and features objects for splits

# Training
y_train = train.iloc[:, 0]
X_train = train.drop(columns = ['Claim PD Number of Paid Items'])

# Test
y_test = test1.iloc[:, 0]
X_test = test1.drop(columns = ['Claim PD Number of Paid Items'])

# Validation
y_pred = pred.iloc[:, 0]
X_pred = pred.drop(columns = ['Claim PD Number of Paid Items'])

We can now build the XGBoost model!

In [14]:
# XGBoost model
xg_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    colsample_bytree=0.3,
    subsample=0.8,
    learning_rate=0.05,
    max_depth=5,
    alpha=1,
    n_estimators=800,
    random_state=42,
    n_jobs=-1
)

xg_reg.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.3
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

Let's now evaluate the model on the test set.

In [15]:
# Evaluating model on test data
predictions = xg_reg.predict(X_test)
errors = abs(predictions - y_test)
mape = 100 * np.mean(errors / y_test)
mse=mean_squared_error(y_test,predictions)
RMSE=np.sqrt(mse)

print("XGBOOST model")
print("mape value for test set",mape)
print("mse value for test set",mse)
print("RMSE value for test set",RMSE)

XGBOOST model
mape value for test set 6.346000635277537
mse value for test set 2643413760.0
RMSE value for test set 51414.13968938895


We can then evaluate the model on the validation set.

In [16]:
# Evaluating the model on test data
predictions = xg_reg.predict(X_pred)
errors = abs(predictions - y_pred)
mape = 100 * np.mean(errors / y_pred)
mse=mean_squared_error(y_pred,predictions)
RMSE=np.sqrt(mse)

print("XGBOOST model")
print("mape value for validation set",mape)
print("mse value for validation set",mse)
print("RMSE value for validation set",RMSE)

XGBOOST model
mape value for validation set 9.599815113229141
mse value for validation set 6689145856.0
RMSE value for validation set 81787.198607117


## LightGBM model

We can use the same dataset to apply a LightGBM model to compare against XGBoost.

In [17]:
# LightGBM model
lgb_reg = LGBMRegressor(n_estimators=100, random_state=42)
lgb_reg.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000030 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 206
[LightGBM] [Info] Number of data points in the train set: 132, number of used features: 6
[LightGBM] [Info] Start training from score 644750.007576
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [18]:
# Evaluating the model on test data
predictions = lgb_reg.predict(X_test)
errors = abs(predictions - y_test)
mape = 100 * np.mean(errors / y_test)
mse=mean_squared_error(y_test,predictions)
RMSE=np.sqrt(mse)

print("LIGHTGBM model")
print("mape value for test set",mape)
print("mse value for test set",mse)
print("RMSE value for test set",RMSE)

LIGHTGBM model
mape value for test set 4.408660958503431
mse value for test set 1542446885.0039437
RMSE value for test set 39273.99756841597


In [19]:
# Evaluating the model on test data
predictions = lgb_reg.predict(X_pred)
errors = abs(predictions - y_pred)
mape = 100 * np.mean(errors / y_pred)
mse=mean_squared_error(y_pred,predictions)
RMSE=np.sqrt(mse)

print("LIGHTGBM model")
print("mape value for validation set",mape)
print("mse value for validation set",mse)
print("RMSE value for validation set",RMSE)

LIGHTGBM model
mape value for validation set 7.930807785604937
mse value for validation set 5026534719.486857
RMSE value for validation set 70898.05864399149


## Random Forest

We can also create a Random Forest model to go alongside this.

In [20]:
# Random Forest model
regr = RandomForestRegressor(n_estimators=100, random_state=42)
regr.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [21]:
# Evaluating the model on test data
predictions = regr.predict(X_test)
errors = abs(predictions - y_test)
mape = 100 * np.mean(errors / y_test)
mse=mean_squared_error(y_test,predictions)
RMSE=np.sqrt(mse)

print("RANDOM FOREST model")
print("mape value for test set",mape)
print("mse value for test set",mse)
print("RMSE value for test set",RMSE)

RANDOM FOREST model
mape value for test set 3.4642244207431174
mse value for test set 836166745.344929
RMSE value for test set 28916.54795000484


In [22]:
# Evaluating the model on test data
predictions = regr.predict(X_pred)
errors = abs(predictions - y_pred)
mape = 100 * np.mean(errors / y_pred)
mse=mean_squared_error(y_pred,predictions)
RMSE=np.sqrt(mse)

print("RANDOM FOREST model")
print("mape value for validation set",mape)
print("mse value for validation set",mse)
print("RMSE value for validation set",RMSE)

RANDOM FOREST model
mape value for validation set 4.557453768061382
mse value for validation set 1954035109.357537
RMSE value for validation set 44204.46933690684


## SARIMA model

In [23]:
#function to evaluate the model.
def evaluate(model, test_features, test_labels):
    predictions = model.predict(test_features)
    errors = abs(predictions - test_labels)
    mape = 100 * np.mean(errors / test_labels)
    accuracy = 100 - mape
    mse=mean_squared_error(test_labels,predictions)
    RMSE=np.sqrt(mse)
    print('Model Performance')
    print('Average Error: {:0.4f} degrees.'.format(np.
mean(errors)))
    print('Accuracy = {:0.2f}%.'.format(accuracy))
    print('RMSE = {:0.2f}'.format(RMSE))
    return accuracy,predictions,RMSE


models=[xg_reg, lgb_reg, regr]
model_name=['XGBoost', 'LightGBM', 'RandomForest']
model_RMSE=[]
model_predictions=[]
for item in models:
    base_accuracy,predictions,RMSE=evaluate(item,X_test,y_test)
    model_RMSE.append(RMSE)
    model_predictions.append(predictions)
r=model_RMSE.index(min(model_RMSE))
best_model_predictions=model_predictions[r]
best_model_name=model_name[r]
best_model=models[r]

print('Best Model:')
print(best_model_name)

print('Model Object:')
print(best_model)
print('Predictions:')
print(best_model_predictions)

Model Performance
Average Error: 47168.5691 degrees.
Accuracy = 93.65%.
RMSE = 51414.14
Model Performance
Average Error: 33045.3067 degrees.
Accuracy = 95.59%.
RMSE = 39274.00
Model Performance
Average Error: 25456.7671 degrees.
Accuracy = 96.54%.
RMSE = 28916.55
Best Model:
RandomForest
Model Object:
RandomForestRegressor(random_state=42)
Predictions:
[680812.34 653416.73 697336.45 679787.13 673020.77 676909.92 669216.71
 675734.25 717977.02 636173.54 629202.63 712061.39 648166.88 669979.23
 674889.64 695564.17 727746.92 702972.52 708721.84 725524.71 755015.34
 705395.72 700029.79 757061.53 708167.35 749669.96 755735.08 738436.2
 754110.09 723174.47 740833.99 743048.21 753950.87 725103.85 739213.09
 731520.1  741931.58 757257.06]


In [24]:
# Add day column
X_test['day'] = 1

#Plot timeseries
y_test=pd.DataFrame(y_test)
y_test['predictions']=predictions
X_test['datetime']=pd.to_datetime(X_test[['year','month','day']])
y_test['datetime']=X_test['datetime']
y_test=y_test.sort_values(by='datetime')
trace0 = go.Scatter(x=y_test['datetime'].astype(str), 
y=y_test['Claim PD Number of Paid Items'].values, opacity = 0.8, name='actual_value')
trace1 = go.Scatter(x=y_test['datetime'].astype(str), y=y_test['predictions'].values, opacity = 0.8, name='prediction')
layout = dict(
    title= "Prediction vs actual:",
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                 dict(count=1, label='1m', step='month', 
stepmode='backward'),
                 dict(count=6, label='6m', step='month', 
stepmode='backward'),
                 dict(count=12, label='12m', step='month', 
stepmode='backward'),
                dict(step='all')
            ])
        ),
        rangeslider=dict(visible = True),
        type='date'
    )
)
fig = dict(data= [trace0,trace1], layout=layout)
iplot(fig)